# Ariel 2025 — Notebook 3: Deep Sequence Models

So sánh nhóm **deep learning trên light curve thô**: CNN1D, TCN, LSTM, GRU, Transformer, Autoencoder+MLP.

Khác với Notebook 2 (features dạng bảng), các model này nhận tensor chuỗi `[planets, time, channels]` với channels = FGS white-light + AIRS light curves (bin theo bước sóng). Mỗi model có head `(mu, log_sigma)` và được train bằng Gaussian NLL — xuất σ trực tiếp.

> **Khuyến nghị bật GPU** (Settings → Accelerator → GPU). Deep models dễ overfit khi ít planet — dùng như nhóm baseline so sánh, không nhất thiết là model chính.

In [ ]:
# === Setup: clone running branch and make ariel_ml importable ===
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/Jun1801/ML_IT3190E_Project.git"
CLONE_DIR = Path("/kaggle/working/ML_IT3190E_Project")
if not CLONE_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "running", "--single-branch", REPO_URL, str(CLONE_DIR)],
        check=True,
    )
    print("Cloned branch 'running' →", CLONE_DIR)
else:
    subprocess.run(["git", "-C", str(CLONE_DIR), "pull"], check=True)
    print("Pulled latest →", CLONE_DIR)

CANDIDATE_SRC = [
    str(CLONE_DIR / "src"),
    "/kaggle/input/ariel-ml-src/src",
    "/kaggle/usr/lib/ariel_ml",
    "src", "../src",
]
for _p in CANDIDATE_SRC:
    if Path(_p).exists():
        sys.path.insert(0, _p); print("Using ariel_ml from:", _p); break
else:
    print("WARNING: ariel_ml source not found.")

DATA_ROOT = Path("/kaggle/input/ariel-data-challenge-2025")
OUTPUT_DIR = Path("/kaggle/working"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WEIGHTS_DIR = OUTPUT_DIR / "weights"; WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Cấu hình

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from ariel_ml.config import PreprocessConfig, DeepModelConfig
from ariel_ml.io import ArielDataRepository
from ariel_ml.pipeline import ArielPreprocessFeaturePipeline
from ariel_ml.sequence_dataset import build_sequence_dataset
from ariel_ml.training import evaluate_prediction, make_train_validation_split
from ariel_ml.deep_models import (
    CNN1DRegressor, TCNRegressor, LSTMRegressor, GRURegressor,
    TransformerSequenceRegressor, AutoencoderMLPRegressor,
)

LIMIT = 300            # số planet (None = full); deep models cần nhiều dữ liệu
TIME_STEPS = 128       # số điểm thời gian sau binning
WAVELENGTH_BINS = 64   # số channel AIRS sau khi gộp bước sóng (None = giữ hết ~356)
EPOCHS = 80
BATCH_SIZE = 32
HIDDEN = 128
RANDOM_STATE = 42


## 2. Dựng tensor chuỗi từ raw light curves
Tái dùng pipeline calibrate → extract → normalize → detrend qua `build_sequence_dataset`.

In [ ]:
from ariel_ml.config import DatasetConfig
repository = ArielDataRepository(DatasetConfig(data_root=DATA_ROOT))
pipeline = ArielPreprocessFeaturePipeline(
    PreprocessConfig(target_time_bins=TIME_STEPS, apply_cds=True, detrend_degree=2, smooth_window=5)
)

dataset = build_sequence_dataset(
    repository, pipeline, "train",
    limit=LIMIT, wavelength_bins=WAVELENGTH_BINS, on_error="skip",
)
print("Sequence tensor:", dataset.x.shape, "| planets kept:", len(dataset.planet_ids),
      "| failures:", len(dataset.failures))


## 3. Align với target + chuẩn hóa per-channel

In [ ]:
targets = pd.read_csv(DATA_ROOT / "train.csv")
targets["planet_id"] = targets["planet_id"].astype(str)
target_cols = [c for c in targets.columns if c != "planet_id"]
tmap = targets.set_index("planet_id")[target_cols]

mask = [pid in tmap.index for pid in dataset.planet_ids]
X = dataset.x[np.array(mask)]
ids = [pid for pid, keep in zip(dataset.planet_ids, mask) if keep]
Y = tmap.loc[ids].to_numpy(dtype=float)
print("X:", X.shape, "| Y:", Y.shape)

# split by planet, then standardize channels using TRAIN stats only
tr, va = make_train_validation_split(n_samples=X.shape[0], validation_fraction=0.2, random_state=RANDOM_STATE)
mean = X[tr].mean(axis=(0, 1), keepdims=True)
std = X[tr].std(axis=(0, 1), keepdims=True) + 1e-8
Xn = (X - mean) / std
print("train:", len(tr), "val:", len(va))


## 4. Train & so sánh các kiến trúc deep sequence

In [ ]:
ARCHITECTURES = {
    "cnn1d": CNN1DRegressor,
    "tcn": TCNRegressor,
    "lstm": LSTMRegressor,
    "gru": GRURegressor,
    "transformer": TransformerSequenceRegressor,
    "autoencoder_mlp": AutoencoderMLPRegressor,
}

rows = []
for name, cls in ARCHITECTURES.items():
    cfg = DeepModelConfig(epochs=EPOCHS, batch_size=BATCH_SIZE, hidden_size=HIDDEN,
                          random_state=RANDOM_STATE, device="auto")
    model = cls(cfg).fit(Xn[tr], Y[tr])
    weight_path = WEIGHTS_DIR / f"{name}_weights.pt"
    model.save_weights(weight_path)
    print(f"  → weights saved: {weight_path}")
    pred = model.predict(Xn[va])
    ev = evaluate_prediction(Y[va], pred)
    rows.append({"model": name, "ariel_gll_score": ev.ariel_gll_score,
                 "rmse_mean": ev.rmse_mean, "gaussian_nll": ev.gaussian_nll,
                 "coverage_1sigma": ev.coverage_1sigma})
    print(f"{name:16s} GLL={ev.ariel_gll_score:.4f}  RMSE={ev.rmse_mean:.3e}  cov1={ev.coverage_1sigma:.3f}")

deep_table = pd.DataFrame(rows).sort_values("ariel_gll_score", ascending=False)
deep_table.to_csv(OUTPUT_DIR / "exp5_deep_sequence.csv", index=False)
deep_table

In [ ]:
t = deep_table.sort_values("ariel_gll_score")
plt.figure(figsize=(8, 4))
plt.barh(t["model"], t["ariel_gll_score"], color="seagreen")
plt.xlabel("Ariel GLL score (higher = better)")
plt.title("Exp 5 — deep sequence models (held-out split)")
plt.tight_layout(); plt.show()


## 5. Ghi chú cho báo cáo
- Deep sequence models học pattern ingress/egress trực tiếp từ light curve, không cần feature engineering thủ công.
- Nhưng **dễ overfit** khi số planet nhỏ; cần nhiều dữ liệu + regularization + GPU.
- So sánh trực tiếp với nhóm physics-based ML (Notebook 2): thường Bayesian Ridge + PHC ổn định và calibrate σ tốt hơn trên ít dữ liệu, trong khi deep models có thể vượt khi dữ liệu lớn.
- Bảng `exp5_deep_sequence.csv` dùng chung trục metric (Ariel GLL) với các experiment khác để so sánh công bằng.